# Comandos útiles del banco de ensayo

Cada celda se ejecuta desde el notebook, sin importar en qué carpeta se haya abierto: los comandos corren dentro del repositorio con el entorno del proyecto (`uv run`).

**Orden para energizar:** zapatilla apagada → USB del relé → USB de la ESP32 → encender la zapatilla.
**Para apagar, al revés:** primero cortar la zapatilla y después desenchufar los USB.

Ejecutar primero la celda de **Configuración**.

## Configuración

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path(r"C:\Users\valfe\OneDrive\SoftwareParaEnsayosDeSeguridadElectricaPortatil")
ENV = {
    **os.environ,
    "PYTHONPATH": "src",
    "PYTHONIOENCODING": "utf-8",
    "UV_LINK_MODE": "copy",  # OneDrive no admite los hardlinks que usa uv
}


def correr(*args: str, timeout: float = 120) -> int:
    """Ejecuta un comando dentro del repositorio con el entorno del proyecto.

    Args:
        *args: Comando y argumentos (por ejemplo "python", "scripts/x.py").
        timeout: Tiempo máximo en segundos.

    Returns:
        Código de salida del comando.
    """
    r = subprocess.run(
        ["uv", "run", "--no-sync", *args],
        cwd=REPO, env=ENV, capture_output=True, text=True, encoding="utf-8", timeout=timeout,
    )
    print(r.stdout, r.stderr, sep="")
    return r.returncode


def correr_python(codigo: str, timeout: float = 120) -> int:
    """Ejecuta código Python con el paquete ensayos disponible."""
    return correr("python", "-c", codigo, timeout=timeout)


print("Repositorio:", REPO, "| existe:", REPO.exists())

## ESP32

### Detectar la ESP32
Lista los puertos serie e indica si el programa reconoce la placa (debería ser COM4).

In [ ]:
correr("python", "scripts/diagnostico_esp32.py")

### Leer el PZEM1
Con la zapatilla apagada responde `PZEM1:ERROR`; con 220 V, unos 230 V.

In [ ]:
correr("python", "scripts/diagnostico_esp32.py", "--pzem", "1")

## Relé USB

### Ver el estado (solo lectura)
No conmuta nada.

In [ ]:
correr("python", "scripts/diagnostico_rele.py")

### Probar los relés
Enciende el relé 1 y después el relé 2, `SEGUNDOS` cada uno, y siempre deja todo apagado al terminar.

⚠️ **Apagar la zapatilla antes**: con 220 V, el relé 1 cierra el contactor y el tomacorriente del DUT queda energizado.

In [ ]:
SEGUNDOS = 2
correr("python", "scripts/diagnostico_rele.py", "--probar", str(SEGUNDOS))

## Contactor

### Accionar el contactor (relé 1)
Cierra el contactor durante `SEGUNDOS` y lo vuelve a abrir. Los relés se apagan aunque haya un error.

⚠️ **Mientras está cerrado, el tomacorriente del DUT tiene 220 V.** No tocar el tomacorriente ni sus cables.

In [ ]:
SEGUNDOS = 2
correr_python(f"""
import time
from ensayos.hardware.relay_controller import RelayController
rc = RelayController()
serie = rc.get_active_serial()
try:
    rc.turn_on(serie, 1)
    print("Contactor cerrado, estado:", rc.get_state(serie))
    time.sleep({SEGUNDOS})
finally:
    if rc.turn_off_all(serie):
        print("Contactor abierto (relés confirmados apagados)")
    else:
        print("¡ATENCIÓN! No se pudo confirmar el apagado: cortar la zapatilla")
""")

### Apagar todos los relés (emergencia)
Si algo queda encendido, esta celda abre todos los relés y, con ellos, el contactor.

In [ ]:
correr_python("""
from ensayos.hardware.relay_controller import RelayController
rc = RelayController()
serie = rc.get_active_serial()
if rc.turn_off_all(serie):
    print("Relés confirmados apagados")
else:
    print("¡ATENCIÓN! No se pudo confirmar el apagado: cortar la zapatilla")
""")

## Medición de una carga

Cierra el contactor, lee el PZEM1 durante `SEGUNDOS` y vuelve a abrirlo.
Enchufar la carga en el tomacorriente del DUT **antes**, con la zapatilla apagada.

In [ ]:
SEGUNDOS = 5
correr_python(f"""
import time
from ensayos.hardware.esp32_client import ESP32Client
from ensayos.hardware.relay_controller import RelayController
rc = RelayController()
serie = rc.get_active_serial()
esp = ESP32Client()
esp.connect()
try:
    rc.turn_on(serie, 1)
    time.sleep(1)  # estabilización
    for _ in range({SEGUNDOS}):
        if rc.get_state(serie) != 1:
            # La placa se reinició (relés apagados) o se desconectó
            print("Aviso: la placa de relés se reinició o desconectó; el contactor se abrió.")
            break
        print(esp.read_pzem(1))
        time.sleep(1)
finally:
    ok = rc.turn_off_all(serie)
    esp.disconnect()
    print("Contactor abierto" if ok else "¡ATENCIÓN! No se pudo confirmar el apagado: cortar la zapatilla")
""", timeout=60)

## Interfaz gráfica

Abre la GUI en pantalla completa (se cierra con **Alt+F4**). Enchufar la ESP32 y el relé **antes** de abrirla.

In [ ]:
subprocess.Popen(
    ["uv", "run", "--extra", "gui", "python", "-m", "ensayos.gui.main"],
    cwd=REPO, env=ENV,
)

## Tests

In [ ]:
correr("python", "-m", "pytest", "-q", timeout=300)

## Firmware de la ESP32

Compila y carga `firmware/esp32_pzem` en COM4. Cortar la zapatilla antes.

In [ ]:
PIO = Path.home() / ".local" / "bin" / "pio.exe"
r = subprocess.run(
    [str(PIO), "run", "-d", "firmware/esp32_pzem", "-t", "upload", "--upload-port", "COM4"],
    cwd=REPO,
    env={**ENV, "PLATFORMIO_WORKSPACE_DIR": str(Path(os.environ["LOCALAPPDATA"]) / "pio-ws" / "esp32_pzem")},
    capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=900,
)
print(r.stdout[-3000:], r.stderr[-2000:], sep="")